# 📈 Mi primera regresión lineal (versión sencilla)
**Curso de IA · Bloque de Machine Learning · Sesión 2 (versión básica)**

---

## ¿Qué vamos a hacer hoy?

**Predecir el precio de una vivienda** a partir de sus características (metros cuadrados, antigüedad, distancia al centro).

Y lo haremos con la herramienta más sencilla del Machine Learning: la **regresión lineal**.

## La idea en 30 segundos

Seguro que has visto que **cuanto más grande es un piso, más caro es**. Si dibujamos los pisos como puntos (metros en horizontal, precio en vertical), los puntos **suben de izquierda a derecha**.

La regresión lineal **dibuja la recta que mejor pasa por esos puntos**. Después, para predecir el precio de un piso nuevo, **miramos en la recta**.

La recta tiene esta fórmula:

```
precio = b + a × metros
```

- **a** (la *pendiente*): cuánto **sube el precio por cada metro más**.
- **b** (el *punto de partida*): dónde empieza la recta.

> 💡 No hay más. Todo lo de hoy es: **dibujar**, **ajustar la recta**, **medir si acierta** y **usarla**.

## Los pasos de hoy

| Apartado | Paso | En una frase |
|---|------|--------------|
| 1-2 | Preparar herramientas y datos | Importar librerías y crear el CSV |
| 3 | Cargar y mirar los datos | ¿Cuántos hay? ¿Cómo son? |
| 4 | Dibujarlos | ¿Suben los puntos? |
| 5 | Separar entrenamiento y prueba | Guardar unos pisos "para el examen" |
| 6 | Entrenar la recta | Que el ordenador la calcule |
| 7 | Medir los errores | ¿Cuánto se equivoca? |
| 8 | Predecir | Calcular el precio de un piso nuevo |
| 9 | Usar más variables | ¿Mejora si miramos más cosas? |
| 10 | Resumen | La receta para recordar |

## Cómo usar este cuaderno

- Ejecuta cada celda de código con **Shift + Enter** (o con el botón ▶).
- Hazlo **en orden**, de arriba abajo.
- Si algo se estropea: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.
- Antes de ejecutar cada celda, **lee el texto de encima** y piensa qué crees que va a salir.

> 👩‍🏫 **Nota para el profesor**
>
> Esta es una **versión simplificada** del cuaderno de regresión lineal de viviendas. Se ha quitado a propósito todo lo que no es imprescindible: **no hay Pipelines, ni nulos, ni escalado, ni validación cruzada, ni comparación con otros modelos**. Los datos vienen limpios para que la atención se quede en la idea central.
>
> **Duración orientativa: unas 3 horas** con los ejercicios.
>
> Cuando los alumnos dominen esto, el cuaderno completo (con nulos, `Pipeline`, sobreajuste y comparación con árboles) es el siguiente paso natural.

## 1. Preparar las herramientas

Importamos tres librerías que ya conocemos:

- **pandas** → para trabajar con tablas (como Excel)
- **numpy** → para cálculos con números
- **matplotlib** → para dibujar gráficos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Todo listo ✅")

## 2. Crear el fichero de datos

Para no depender de ningún fichero externo, esta celda **fabrica** un CSV con **200 viviendas** y lo guarda como `viviendas.csv`.

| Columna | Qué significa |
|---|---|
| `m2` | Metros cuadrados |
| `antiguedad` | Años que tiene el edificio |
| `distancia_centro` | Distancia al centro de la ciudad (km) |
| `precio` | Precio de la vivienda en euros ← **lo que queremos predecir** |

> 💡 **No hace falta entender el código de esta celda.** Es solo la "fábrica" de datos. Lo importante empieza en el paso 3.

In [ ]:
rng = np.random.default_rng(7)    # esto hace que todos obtengamos los mismos datos
n = 200

m2 = rng.uniform(40, 200, n).round(0)
antiguedad = rng.integers(0, 61, n)
distancia_centro = rng.uniform(0.5, 25, n).round(1)
ruido = rng.normal(0, 20000, n)                     # un poco de "azar", como en la vida real

precio = (80000 + 2200 * m2 - 600 * antiguedad - 2000 * distancia_centro + ruido).round(-2)

datos = pd.DataFrame({
    "m2": m2,
    "antiguedad": antiguedad,
    "distancia_centro": distancia_centro,
    "precio": precio,
})

datos.to_csv("viviendas.csv", index=False)
print("✅ Fichero 'viviendas.csv' creado")

## 3. Cargar los datos y mirarlos

Lo primero, siempre: **mirar con qué datos trabajamos**.

`pd.read_csv()` lee el fichero y lo guarda en una tabla (en programación la llamamos `df`, de *DataFrame*).

In [ ]:
df = pd.read_csv("viviendas.csv")

# head() muestra las primeras filas
df.head()

Cada fila es **una vivienda**. Cada columna es **una característica**.

¿Cuántas viviendas hay y cuántas columnas? `shape` nos lo dice (filas, columnas):

In [ ]:
print("Filas y columnas:", df.shape)

### Un resumen con números: `describe()`

Nos da, para cada columna, valores como el **mínimo**, el **máximo** y la **media**.

In [ ]:
df.describe().round(1)

👀 **Cómo leerlo (mira la columna `precio`):**

| Fila | Qué significa |
|---|---|
| `count` | Cuántas viviendas hay (200) |
| `mean` | El precio **medio** |
| `min` | El piso **más barato** |
| `max` | El piso **más caro** |

Fíjate también en `m2`: ¿cuál es el piso más pequeño y cuál el más grande?

## 4. Dibujar los datos

**Antes de usar ningún modelo, hay que dibujar los datos.** Un gráfico de puntos (*scatter*) pone **un punto por vivienda**: metros en el eje horizontal y precio en el vertical.

In [ ]:
plt.scatter(df["m2"], df["precio"], alpha=0.6)
plt.xlabel("Metros cuadrados")
plt.ylabel("Precio (€)")
plt.title("Precio según los metros cuadrados")
plt.show()

👀 **¿Qué ves?** Los puntos **suben de izquierda a derecha**: cuanto más grande el piso, más caro. No forman una línea perfecta (cada piso es distinto), pero la **tendencia es clara**.

Esa tendencia es justo lo que la regresión lineal va a capturar con una recta.

### Medir la relación con un número: la correlación

La **correlación** dice, con un número, **cuánto se parecen dos columnas en su subida y bajada**:

| Valor | Significa |
|---|---|
| Cerca de **1** | Cuando una sube, la otra **sube** (relación fuerte) |
| Cerca de **0** | No hay relación |
| Cerca de **−1** | Cuando una sube, la otra **baja** |

In [ ]:
df.corr()["precio"].round(2)

👀 **Mira los números:**
- `m2` tiene una correlación **alta y positiva**: más metros → más precio.
- `antiguedad` y `distancia_centro` son **negativas** (y mucho más débiles): más años o más lejos del centro → **menos** precio. Tiene sentido.

> ⚠️ Una correlación alta **no prueba** que una cosa cause la otra. Solo dice que **van juntas**.

## 5. Separar datos de entrenamiento y de prueba

**Analogía del examen 📝:** si un profesor da a los alumnos las preguntas y respuestas del examen para estudiar, y luego les pone ese mismo examen, todos sacarán buena nota... pero **eso no demuestra que hayan aprendido**.

Con los modelos pasa igual. Dividimos los pisos en dos grupos:

| Grupo | Para qué sirve | Analogía |
|---|---|---|
| **Entrenamiento** (80 %) | El modelo **aprende** de estos pisos | Los apuntes |
| **Prueba** (20 %) | Comprobamos si **acierta con pisos que nunca ha visto** | El examen |

Primero decidimos **qué usamos para predecir** (la `X`) y **qué queremos predecir** (la `y`). Empezamos con **una sola variable**: los metros.

In [ ]:
X = df[["m2"]]        # lo que usamos para predecir (¡con DOBLE corchete!)
y = df["precio"]      # lo que queremos predecir

print("X tiene", X.shape[0], "filas y", X.shape[1], "columna")

Ahora separamos con `train_test_split`:

- `test_size=0.2` → el 20 % de los pisos se guarda para el examen.
- `random_state=84` → hace que el "sorteo" sea siempre el mismo (así todos obtenemos lo mismo).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=84)

print("Pisos para entrenar:", len(X_train))
print("Pisos para el examen:", len(X_test))

> 🔒 **Regla de oro:** los pisos del examen **no se tocan** hasta el momento de medir. Solo se usan para evaluar.

## 6. Entrenar la recta

Ahora viene lo importante, y es **muy corto**. Con scikit-learn, entrenar un modelo siempre son **tres líneas**:

```python
modelo = LinearRegression()          # 1. Crear el modelo
modelo.fit(X_train, y_train)         # 2. Entrenar (que aprenda la recta)
modelo.predict(X_nuevo)              # 3. Predecir
```

`fit` significa "ajustar": el ordenador **busca la recta que mejor pasa por los puntos de entrenamiento**.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("✅ Modelo entrenado")

### ¿Qué recta ha encontrado?

El modelo guarda los dos números de la recta:
- `coef_` → la **pendiente** (a)
- `intercept_` → el **punto de partida** (b)

In [ ]:
a = modelo.coef_[0]
b = modelo.intercept_

print(f"Pendiente (a): {a:,.0f}")
print(f"Punto de partida (b): {b:,.0f}")
print()
print(f"Recta: precio = {b:,.0f} + {a:,.0f} × metros")

👀 **Cómo leerlo en lenguaje normal:**

> *"Por cada metro cuadrado más, el precio sube, de media, esa cantidad de euros."*

El **punto de partida** es solo un número necesario para colocar la recta; no hay que interpretarlo (no existen pisos de 0 m²).

### Dibujar la recta

Pintamos los pisos de entrenamiento y encima la recta que ha aprendido el modelo:

In [ ]:
plt.scatter(X_train["m2"], y_train, alpha=0.5, label="Pisos (entrenamiento)")

# Para dibujar la recta, pedimos al modelo el precio de pisos de 40 y de 200 m²
metros = pd.DataFrame({"m2": [40, 200]})
plt.plot(metros["m2"], modelo.predict(metros), color="red", linewidth=3, label="Recta del modelo")

plt.xlabel("Metros cuadrados")
plt.ylabel("Precio (€)")
plt.title("La recta que mejor pasa por los puntos")
plt.legend()
plt.show()

👀 La línea roja atraviesa la nube de puntos por el medio: es la **mejor recta posible** para estos datos.

## 7. ¿Se equivoca mucho? Medir los errores

Ahora usamos los **pisos del examen** (los que el modelo no ha visto). Le pedimos que prediga su precio y comparamos con el **precio real**.

In [ ]:
predicciones = modelo.predict(X_test)

comparacion = pd.DataFrame({
    "m2": X_test["m2"].values,
    "precio real": y_test.values,
    "precio predicho": predicciones.round(-2),
})
comparacion["error"] = (comparacion["precio real"] - comparacion["precio predicho"]).abs()
comparacion["error %"] = (((comparacion["precio real"] - comparacion["precio predicho"])).abs()/comparacion["precio real"]).round(4)

comparacion.head(8)

👀 Fíjate: **el modelo no acierta exactamente**, pero se acerca. Algunas veces se pasa y otras se queda corto. La columna `error` dice **cuántos euros se ha equivocado** en cada piso.

### Una medida resumen: el error medio (MAE)

Para saber **cuánto se equivoca en general**, calculamos la **media de esos errores**. Se llama **MAE** (*Mean Absolute Error*, error absoluto medio):

> *"De media, el modelo se equivoca en X euros."*

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, predicciones)
print(f"Error medio (MAE): {mae:,.0f} €")
print(f"Precio medio de los pisos de prueba: {y_test.mean():,.0f} €")
print(f"Porcentaje del error medio: {((1-(y_test.mean()-mae)/y_test.mean())*100):,.2f}")

### ¿Es mucho o es poco? Hay que compararlo con algo

Un error de unos 18.000 € no dice nada por sí solo. Comparémoslo con **un modelo "tonto"** que **siempre predice el precio medio**, sin mirar ni los metros:

In [ ]:
precio_medio = y_train.mean()                     # el modelo tonto siempre dice esto
errores_tonto = (y_test - precio_medio).abs()
mae_tonto = errores_tonto.mean()

print(f"Error del modelo tonto (siempre la media): {mae_tonto:,.0f} €")
print(f"Error de nuestra recta                    : {mae:,.0f} €")

👀 **Nuestra recta se equivoca bastante menos** que el modelo tonto. Eso significa que **sí ha aprendido algo útil**: los metros sirven para predecir el precio.

> 🎯 **Idea clave:** un modelo solo es bueno si **mejora a uno "tonto"**. Siempre hay que compararse con algo.

### Otra medida: el R²

El **R²** (se lee "erre cuadrado") es una nota entre 0 y 1 que dice **qué parte de las diferencias de precio explica el modelo**:

| R² | Significa |
|---|---|
| **0** | El modelo no mejora a predecir siempre la media (el "tonto") |
| **0,5** | Explica la mitad de lo que hace variar el precio |
| **1** | Predicción perfecta |

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, predicciones)
print(f"R² = {r2:.2f}   (el modelo explica el {r2 * 100:.0f} % de la variación del precio)")

👀 **Un R² alto** quiere decir que **solo con los metros** ya explicamos gran parte del precio. El resto lo explican otras cosas (antigüedad, ubicación...) y el azar.

## 8. Predecir el precio de un piso nuevo

Es **para lo que sirve todo esto**. Preguntamos por un piso de **100 m²**.

> ⚠️ Hay que dárselo con el mismo formato que usamos al entrenar: **una tabla con una columna `m2`**.

In [ ]:
piso_nuevo = pd.DataFrame({"m2": [100]})

precio_estimado = modelo.predict(piso_nuevo)[0]
print(f"Un piso de 100 m² vale, según el modelo, unos {precio_estimado:,.0f} €")
print(f"(Recuerda que de media nos equivocamos en unos {mae:,.0f} €)")

Probemos con varios tamaños a la vez:

In [ ]:
varios = pd.DataFrame({"m2": [50, 80, 100, 150, 200]})
resultado = varios.copy()
resultado["precio estimado (€)"] = modelo.predict(varios).round(-2)
resultado

## 9. ¿Y si miramos más cosas? Varias variables

Hasta ahora **solo hemos mirado los metros**. Pero el precio también depende de la **antigüedad** y de la **distancia al centro**. Vamos a darle al modelo las **tres** variables.

El código es **casi idéntico**; solo cambia qué columnas usamos en `X`:

In [ ]:
X2 = df[["m2", "antiguedad", "distancia_centro"]]     # ahora, tres columnas

# Separamos de nuevo (con el mismo random_state, así los pisos de prueba son los mismos)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.3, random_state=84)

modelo2 = LinearRegression()
modelo2.fit(X2_train, y2_train)

predicciones2 = modelo2.predict(X2_test)
print("✅ Modelo con 3 variables entrenado")

### ¿Qué ha aprendido? Los coeficientes

Ahora hay **un número por cada variable**. Cada uno dice **cuánto cambia el precio al aumentar esa variable en 1 unidad**, manteniendo las demás igual:

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X2.columns,
    "efecto en el precio (€)": modelo2.coef_.round(0),
})
coeficientes

👀 **Cómo leerlo en lenguaje normal:**

- **`m2` (positivo):** cada m² de más **sube** el precio en esa cantidad.
- **`antiguedad` (negativo):** cada año más de antigüedad **baja** el precio.
- **`distancia_centro` (negativo):** cada km más lejos del centro **baja** el precio.

Los signos tienen **mucho sentido**. ¡El modelo ha descubierto por sí solo cómo funciona el precio de las viviendas!

### ¿Mejora con más variables?

Comparamos el modelo de **1 variable** con el de **3 variables**, con las mismas dos medidas:

In [ ]:
X3 = df[["m2", "antiguedad"]]
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y, test_size=0.3, random_state=84)

modelo3 = LinearRegression()
modelo3.fit(X3_train, y3_train)

predicciones3 = modelo3.predict(X3_test)
print("✅ Modelo con 2 variables entrenado")

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X3.columns,
    "efecto en el precio (€)": modelo3.coef_.round(0),
})
coeficientes

In [ ]:
mae3 = mean_absolute_error(y3_test, predicciones3)
r2_3 = r2_score(y3_test, predicciones3)

In [ ]:
X4 = df[["m2", "distancia_centro"]]
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y, test_size=0.3, random_state=84)

modelo4 = LinearRegression()
modelo4.fit(X4_train, y4_train)

predicciones4 = modelo4.predict(X4_test)
print("✅ Modelo con 2 variables entrenado")

coeficientes = pd.DataFrame({
    "variable": X4.columns,
    "efecto en el precio (€)": modelo4.coef_.round(0),
})
display(coeficientes)

mae4 = mean_absolute_error(y4_test, predicciones4)
r2_4 = r2_score(y4_test, predicciones4)

In [ ]:
mae2 = mean_absolute_error(y2_test, predicciones2)
r2_2 = r2_score(y2_test, predicciones2)

tabla = pd.DataFrame({
    "modelo": ["Siempre la media (tonto)", "Recta con 1 variable (m²)", "Con 3 variables", "Con 2 variables (m2, Dist)", "con 2 variables (m2, Ant)"],
    "error medio MAE (€)": [round(mae_tonto), round(mae), round(mae2), round(mae3), round(mae4)],
    "R²": [0.00, round(r2, 2), round(r2_2, 2), round(r2_3, 2), round(r2_4, 2) ],
})
tabla

👀 **Con más variables útiles, el modelo se equivoca algo menos:** el error baja y el R² sube. La mejora es **pequeña** porque los metros ya explicaban casi todo el precio; en otros problemas, añadir variables puede mejorar mucho más.

### Un último gráfico: precio real contra precio predicho

Si el modelo fuera **perfecto**, todos los puntos estarían **sobre la línea roja** (real = predicho). Cuanto más cerca de la línea, mejor.

In [ ]:
plt.scatter(y2_test, predicciones2, alpha=0.7)

minimo = min(y2_test.min(), predicciones2.min())
maximo = max(y2_test.max(), predicciones2.max())
plt.plot([minimo, maximo], [minimo, maximo], color="red", linestyle="--", label="Predicción perfecta")

plt.xlabel("Precio real (€)")
plt.ylabel("Precio predicho (€)")
plt.title("¿Cuánto se parecen las predicciones a la realidad?")
plt.legend()
plt.show()

👀 Los puntos **siguen la diagonal**: el modelo acierta bastante bien. Los que se separan más de la línea son los pisos en los que más se equivoca.

### Predecir un piso con las 3 variables

In [ ]:
piso_nuevo2 = pd.DataFrame({
    "m2": [90],
    "antiguedad": [20],
    "distancia_centro": [5.0],
})

precio_estimado2 = modelo2.predict(piso_nuevo2)[0]
print(f"Piso de 90 m², 20 años, a 5 km del centro: unos {precio_estimado2:,.0f} €")
print(f"(error medio del modelo: {mae2:,.0f} €)")

## 10. Resumen

### La receta (¡sirve para cualquier regresión!)

| Paso | Qué hacemos | Código |
|---|---|---|
| 1 | Leer los datos | `df = pd.read_csv("fichero.csv")` |
| 2 | Mirarlos y dibujarlos | `df.head()`, `df.describe()`, `plt.scatter(...)` |
| 3 | Elegir qué predecir (`y`) y con qué (`X`) | `X = df[["m2"]]`, `y = df["precio"]` |
| 4 | Separar entrenamiento y prueba | `train_test_split(X, y, test_size=0.2, random_state=84)` |
| 5 | Entrenar | `modelo = LinearRegression()` y `modelo.fit(X_train, y_train)` |
| 6 | Predecir | `modelo.predict(X_test)` |
| 7 | Medir | `mean_absolute_error(...)` y `r2_score(...)` |

### Ideas clave para recordar

1. La **regresión lineal** ajusta **una recta** (o un "plano" con varias variables) a los datos.
2. **Siempre dibujar** los datos antes de modelar.
3. **Siempre separar** entrenamiento y prueba, y medir **con los pisos de prueba**.
4. Un modelo es bueno si **mejora al modelo "tonto"** (el que predice siempre la media).
5. **MAE** = "de media me equivoco en X". **R²** = "qué parte del precio explico" (0 a 1).
6. Los **coeficientes** se pueden **leer y explicar**: es la gran ventaja de este modelo.

### 📖 Mini glosario

| Palabra | Significado sencillo |
|---|---|
| **Variable** | Una columna de la tabla |
| **X** | Las variables que usamos para predecir |
| **y** | La variable que queremos predecir |
| **Entrenar (`fit`)** | Que el modelo aprenda de los datos |
| **Predecir (`predict`)** | Usar el modelo para calcular un valor |
| **Coeficiente** | Cuánto cambia el resultado al subir una variable en 1 unidad |
| **MAE** | Error medio, en las mismas unidades que lo que predecimos |
| **R²** | Nota de 0 a 1: qué parte de la variación explica el modelo |

## ✍️ Ejercicios

**Nivel 1 – Para empezar**
1. Predice el precio de un piso de **60 m²** y otro de **180 m²** con el modelo de **una variable**. El segundo tiene el triple de metros: ¿cuesta exactamente el triple? ¿Por qué no? (Pista: mira el punto de partida de la recta.)
2. En el paso 5, cambia `test_size=0.2` por `0.3`. ¿Cuántos pisos quedan para el examen? ¿Cambian mucho el MAE y el R²?
3. Mira otra vez el gráfico de la recta. ¿Qué pasaría si nuestra recta tuviera una pendiente mucho mayor? ¿Los puntos estarían más cerca o más lejos de ella?

**Nivel 2 – Practicar**
4. Entrena un modelo que use **solo `antiguedad`** para predecir el precio. ¿Qué R² saca? ¿Es mejor o peor que con solo los metros? ¿Por qué crees que pasa?
5. Entrena un modelo con **`m2` y `distancia_centro`** (sin la antigüedad). ¿Cuánto empeora el MAE respecto al de 3 variables?
6. Cambia `random_state=84` por otro número (por ejemplo 7). ¿Cambian un poco los resultados? ¿Por qué?

**Nivel 3 – Reto (opcional)**
7. Haz un bucle que pruebe **cada variable por separado** (`m2`, `antiguedad`, `distancia_centro`) y escriba su R². ¿Cuál es la variable más útil por sí sola?

---
### 🎯 ¿Y ahora qué?

Ya sabéis lo esencial: **cargar, dibujar, separar, entrenar, medir y predecir**. En la siguiente sesión añadiremos la **limpieza de datos con nulos**, las **tuberías (Pipeline)** y veremos cuándo un modelo **memoriza en vez de aprender** (sobreajuste).

In [ ]:
varios = pd.DataFrame({"m2": [60, 180]})
resultado = varios.copy()
resultado["precio estimado (€)"] = modelo.predict(varios).round(-2)
resultado

In [ ]:
pisos_nuevo = pd.DataFrame({
    "m2": [60, 180],
    "antiguedad": [20, 20],
    "distancia_centro": [5.0, 5.0]
})

resultado = pisos_nuevo.copy()
resultado["precio estimado (€)"] = modelo2.predict(pisos_nuevo).round(-2)
display(resultado)